# Olist Lakehouse — Exploratory Data Analysis (EDA) Profiling\n\n> **Tác giả:** Anh Tú (Pedagogical Approach) — *Made by Anh Tu - Share to be share*  \n> **Môi trường:** Apache Spark 3.5.0 + PostgreSQL 16 (Chạy trên container `olist_spark_notebook:8888`)  \n> **Mục tiêu:** Phân tích khám phá cấu trúc, độ đầy đủ (% NULL), số lượng phần tử phân biệt (Cardinality) trên cả 2 nguồn: PostgreSQL OLTP và MinIO Landing S3.

## 1. Khởi Tạo SparkSession Tích Hợp PostgreSQL JDBC & MinIO S3\nSpark đóng vai trò là động cơ tính toán phân tán (Distributed Compute Engine), kết nối trực tiếp tới PostgreSQL để đọc dữ liệu giao dịch và kết nối MinIO S3 để đọc các tệp CSV landing.

In [ ]:
import os\nimport json\nimport pandas as pd\nfrom pyspark.sql import SparkSession\n\n# Đọc cấu hình từ file tóm tắt\nwith open('../docs/week02/eda_summary.json', 'r') as f:\n    eda_summary = json.load(f)\n\n# Hiển thị bảng tổng quan 10 thực thể từ 2 nguồn\noverview_data = [\n    {\n        'Thực Thể': name,\n        'Nguồn Lưu Trữ': info['source_type'],\n        'Số Dòng': f"{info['row_count']:,}",\n        'Số Cột': info['column_count']\n    }\n    for name, info in eda_summary.items()\n]\ndf_overview = pd.DataFrame(overview_data)\ndisplay(df_overview)

## 2. Chi Tiết Tỷ Lệ NULL & Phân Bố Thuộc Tính Từng Thực Thể\nXem xét các cột có tỷ lệ khuyết thiếu (% NULL) cao để chuẩn bị thiết lập ngưỡng cho Data Quality Rule Catalog (Task 8).

In [ ]:
# Danh sách các cột có chứa giá trị NULL trên toàn hệ thống\nnull_findings = []\nfor entity, info in eda_summary.items():\n    for col, stats in info['columns'].items():\n        if stats['null_pct'] > 0:\n            null_findings.append({\n                'Thực thể': entity,\n                'Cột': col,\n                'Kiểu dữ liệu': stats['data_type'],\n                'Số dòng NULL': f"{stats['null_count']:,}",\n                'Tỷ lệ NULL (%)': f"{stats['null_pct']}%",\n                'Distinct Count': f"{stats['distinct_count']:,}"\n            })\ndf_nulls = pd.DataFrame(null_findings).sort_values(by='Thực thể')\ndisplay(df_nulls)

## 3. Khám Phá Mối Quan Hệ Chéo Nguồn (Cross-Source Relational Exploration)\nKiểm tra xem các khóa ngoại `product_id` trong bảng `order_items` (PostgreSQL) có tồn tại đầy đủ trong tệp `products` (MinIO Landing) hay không.

In [ ]:
# Đọc trực tiếp để kiểm tra Orphan Product Keys\ndf_items = pd.read_csv('../data_prep/processed/batch_B0/order_items.csv')\ndf_products = pd.read_csv('../data_prep/raw/olist_products_dataset.csv')\n\norphan_products = df_items[~df_items['product_id'].isin(df_products['product_id'])]\nprint(f"Tổng số mặt hàng trong mẻ B0: {len(df_items):,}")\nprint(f"Số dòng order_items trỏ tới product_id không tồn tại: {len(orphan_products):,}")